# Moon phase vs crime/accident (sklearn)
Fetch Supabase data, merge moon phases with daily outcomes, and quantify relationships via linear models.

In [ ]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

load_dotenv()
if not os.getenv("SUPABASE_URL") or not os.getenv("SUPABASE_KEY"):
    raise SystemExit("Missing SUPABASE_URL or SUPABASE_KEY in environment")

supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"])

def fetch_table(name: str, batch_size: int = 1000, limit: int | None = None) -> pd.DataFrame:
    rows, start = [], 0
    while True:
        end = start + batch_size - 1
        resp = supabase.table(name).select("*").range(start, end).execute()
        batch = resp.data or []
        rows.extend(batch)
        if len(batch) < batch_size or (limit and len(rows) >= limit):
            break
        start += batch_size
    if limit:
        rows = rows[:limit]
    return pd.DataFrame(rows)


In [ ]:
crimes_df = fetch_table("chicago_crimes")
accident_df = fetch_table("chicago_accident_cleaned")
moon_df = fetch_table("cleaned_moon_data")

print(f"Crimes: {len(crimes_df)} rows")
print(f"Accidents: {len(accident_df)} rows")
print(f"Moon: {len(moon_df)} rows")

if crimes_df.empty or accident_df.empty or moon_df.empty:
    raise SystemExit("One or more tables are empty; cannot proceed")


In [ ]:
def prep_merged(crimes_df, accident_df, moon_df):
    crimes_daily = crimes_df.groupby("date_only")["crime_count"].sum().reset_index().rename(columns={"date_only": "date"})
    acc_daily = accident_df.groupby("crash_date")["incidents_count"].sum().reset_index().rename(columns={"crash_date": "date"})

    crimes_daily["date"] = pd.to_datetime(crimes_daily["date"])
    acc_daily["date"] = pd.to_datetime(acc_daily["date"])
    moon_df = moon_df.copy()
    moon_df["date"] = pd.to_datetime(moon_df["date"])

    merged = moon_df.merge(crimes_daily, on="date", how="left").merge(acc_daily, on="date", how="left")
    merged = merged.dropna(subset=["avg_phase", "moon_category"]).copy()
    merged["phase_sin"] = np.sin(2 * np.pi * merged["avg_phase"])
    merged["phase_cos"] = np.cos(2 * np.pi * merged["avg_phase"])
    return merged

def fit_and_report(df, target_col):
    X = pd.concat(
        [
            df[["phase_sin", "phase_cos", "avg_phase"]],
            pd.get_dummies(df["moon_category"], prefix="moon", dummy_na=False),
        ],
        axis=1,
    )
    y = df[target_col].fillna(0)

    model = LinearRegression()
    r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
    model.fit(X, y)
    coefs = pd.Series(model.coef_, index=X.columns).sort_values()

    print(f"\n=== {target_col} ===")
    print(f"5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
    print("Top positive weights:\n", coefs.tail(5))
    print("Top negative weights:\n", coefs.head(5))

merged = prep_merged(crimes_df, accident_df, moon_df)
fit_and_report(merged, "crime_count")
fit_and_report(merged, "incidents_count")
